In [1]:
import os, torch, shutil

print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("CUDA_HOME:", os.environ.get("CUDA_HOME"))
print("nvcc:", shutil.which("nvcc"))

CUDA available: True
Torch CUDA version: 11.8
CUDA_HOME: None
nvcc: /home/as5606/miniconda3/envs/hulu-med-clean/bin/nvcc


In [2]:
import os, shutil

candidate_cuda_paths = [
    "/usr/local/cuda-11.8",
    "/usr/local/cuda",
    "/opt/cuda",
]

for p in candidate_cuda_paths:
    nvcc_path = os.path.join(p, "bin", "nvcc")
    print(p, "exists:", os.path.exists(p), "nvcc:", os.path.exists(nvcc_path))

/usr/local/cuda-11.8 exists: False nvcc: False
/usr/local/cuda exists: False nvcc: False
/opt/cuda exists: False nvcc: False


In [3]:
import os, torch, shutil

print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("CUDA_HOME:", os.environ.get("CUDA_HOME"))
print("nvcc:", shutil.which("nvcc"))

CUDA available: True
Torch CUDA version: 11.8
CUDA_HOME: None
nvcc: /home/as5606/miniconda3/envs/hulu-med-clean/bin/nvcc


In [4]:
!bash -lc "module avail cuda 2>&1 | head -80"

-------------------------- /tools/Modules/modulefiles --------------------------
cuda/6.5(default)  cuda/7.5  

Key:
(symbolic-version)  


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_PATH = "ZJU-AI4H/Hulu-Med-4B"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
device_map = "auto" if torch.cuda.is_available() else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=dtype,
    device_map=device_map,
    attn_implementation="sdpa",   # safer than flash_attention_2 unless flash-attn is installed
)

/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-06-15 09:44:12,663] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/as5606/miniconda3/envs/hulu-med-clean/bin/../lib/gcc/x86_64-conda-linux-gnu/14.3.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/as5606/miniconda3/envs/hulu-med-clean/bin/../lib/gcc/x86_64-conda-linux-gnu/14.3.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


config.vision_encoder None
Build from config


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████| 2/2 [00:17<00:00,  8.96s/it]


In [6]:
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.parameters()).device)

CUDA available: True
Model device: cuda:0


In [7]:
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

tokenizer = processor.tokenizer
model.eval()

HulumedQwen3ForCausalLM(
  (model): HulumedQwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_l

In [8]:
def hulumed_generate(conversation, modal="text", max_new_tokens=1024, temperature=0.0):
    inputs = processor(
        conversation=conversation,
        add_system_prompt=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    if "modals" not in inputs:
        inputs["modals"] = [modal]

    model_device = next(model.parameters()).device
    use_cuda = model_device.type == "cuda"

    fixed_inputs = {}
    for k, v in inputs.items():
        if isinstance(v, torch.Tensor):
            if k == "pixel_values":
                dtype = torch.bfloat16 if use_cuda else torch.float32
                fixed_inputs[k] = v.to(device=model_device, dtype=dtype)
            else:
                fixed_inputs[k] = v.to(model_device)
        else:
            fixed_inputs[k] = v

    generation_kwargs = {
        "do_sample": temperature > 0,
        "max_new_tokens": max_new_tokens,
        "use_cache": True,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if temperature > 0:
        generation_kwargs["temperature"] = temperature

    with torch.inference_mode():
        output_ids = model.generate(
            **fixed_inputs,
            **generation_kwargs,
        )

    return processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
        use_think=False,
    )[0].strip()

In [11]:
frame_path = "/home/as5606/Datasets/chole80_framed/cholec80/frames/video01/video01_000002.png"
question = "Is the surgical instrument there?"

conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": {
                    "image_path": frame_path,
                },
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

answer = hulumed_generate(conversation, modal="image", max_new_tokens=256)
print(answer)

Yes


In [12]:
import json
from tqdm import tqdm

# Load test dataset
with open('/home/as5606/Datasets/cholec_formatted_data/cholec80_llava_test.json', 'r') as f:
    test_data = json.load(f)


In [14]:
test_data[0]

{'id': 'video31_frame001200',
 'image': '/home/as5606/Datasets/chole80_framed/cholec80/frames/video31/video31_000049.png',
 'conversations': [{'from': 'human',
   'value': 'is scissors used in preparation?\n<image>'},
  {'from': 'gpt', 'value': 'scissors is not used in preparation'}]}

In [ ]:
# Store results
results = []

print("Generating answers for all questions...")
for item in tqdm(test_data, desc="Processing"):
    # Extract data
    item_id = item['id']
    frame_path = item['image']
    original_question = item['conversations'][0]['value'].replace('\n<image>', '').strip()
    
    # Create conversation for Hulu-Med
    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": {
                        "image_path": frame_path,
                    },
                },
                {
                    "type": "text",
                    "text": original_question,
                },
            ],
        }
    ]
    
    try:
        # Generate answer
        answer = hulumed_generate(conversation, modal="image", max_new_tokens=256)
        
        # Store result
        result = {
            "id": item_id,
            "image": frame_path,
            "question": original_question,
            "answer": answer
        }
        results.append(result)
        
    except Exception as e:
        print(f"Error processing {item_id}: {e}")
        continue

Generating answers for all questions...


Processing:   0%|▏                                                                                       | 15/7652 [00:18<2:40:04,  1.26s/it]


KeyboardInterrupt: 

: 

In [ ]:
# Save to JSON
output_path = '/home/as5606/Datasets/cholec80_hulu_med_answers.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved {len(results)} results to {output_path}")
print(f"Sample result: {results[0] if results else 'No results'}")